<a href="https://colab.research.google.com/github/Swas26/NLP-ESG/blob/main/data_preprocessing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Data Preprocessing
**Pipeline:** ESG PDF reports + NewsAPI articles → cleaned, chunked, combined CSV

**Only needs to be re-run when source data changes.**
The filtering/scoring notebook pulls outputs directly from GitHub.

**Outputs pushed to `data/processed/` on GitHub:**
- `esg_chunks.csv` — chunked text from 12 company ESG PDFs
- `news_articles.csv` — news articles fetched from NewsAPI
- `combined.csv` — merged dataset, input for filtering/scoring notebook

**Before running:**
1. Click the 'key' icon in the left sidebar
2. Add a secret called `GITHUB_TOKEN` with your personal access token
   - Get one at: github.com/settings/tokens → Generate new token (classic) → tick `repo`
3. Toggle **Notebook access** on

In [1]:
# --- SETUP ---
!pip install pdfplumber newsapi-python gdown -q

import os
import re
import pandas as pd
from google.colab import userdata

GITHUB_TOKEN = userdata.get('GITHUB_TOKEN')

FOLDER_ID  = '1SUewYSR3PYYAMUHi9Egqc3Bd8wA0zKeY'
DATA_DIR   = '/content/data'
OUTPUT_DIR = '/content/outputs'
REPO       = 'Swas26/NLP-ESG'
REPO_DIR   = '/content/repo'
GIT_EMAIL  = 'swas.2625@gmail.com'
GIT_NAME   = 'Swas26'

os.makedirs(DATA_DIR, exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)

import gdown
gdown.download_folder(id=FOLDER_ID, output=DATA_DIR, quiet=False)

import torch
print('GPU available:', torch.cuda.is_available())
print('Setup complete ✓')

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 2.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.2/68.2 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.6/6.6 MB 63.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.7/3.7 MB 49.3 MB/s eta 0:00:00


Retrieving folder contents


Processing file 1-qbHDTR_GXkNJSuxW5AZynMF3zzOMEXi AGL.pdf
Processing file 1eZvP7zNSs9SqlMyNeucuurMGp-B4qwSw ANZ
Processing file 17Og3VhIHgiqByuqCXdiXVoA_jtI15Hxz CBA.pdf
Processing file 1EVdVxcyJuHUEuK9Y4VLMAJvk-dwwuilc Coles Group.pdf
Processing file 1BQGJFj2dCAiHCQgJRea9YFn-gxbgQlXF combined.csv
Processing file 1MnbN2U-op5qvUKYCpsyMeXKPyxWBEw2J data_preprocessing.ipynb
Processing file 1k0x71VF13wqgk8sXAqmvAQjDRwaTbePi0bVjwQWS-a4 esg_chunks
Processing file 1RgN0vQbxedUP-YKWYw8Yeou97claEXPu esg_chunks.csv
Processing file 16S4eAPqYulAc0V7hP9uSk4Qy_MybagD- filtered.csv
Processing file 1BAx2TudjBO18BAhFzXytrmoEuAKMvdDH Gucci.pdf
Processing file 12yD29Ro8pu_09Adm0mwIrcH0OxbmgQxw individual_scores.csv
Processing file 1NXWlxi2YsY-S_eG3lM65qm-LgbZOVcGn LVMH.pdf
Processing file 1KOkSDAKtUsPS2mEX24JQHdefVtnBN09A Muji.pdf
Processing file 1UUU7h8YSx2jZQ623SD8OXLN2qxPuBT8M NBN.pdf
Processing file 1Pn_J_UlBce2-r4Mk6TxMwk-2-Ceqbs4r NLP_Assessment_3_Filtering_and_Scoring.ipynb
Processing file 1WXPAmm

Retrieving folder contents completed
Building directory structure
Building directory structure completed
Downloading...
From: https://drive.google.com/uc?id=1-qbHDTR_GXkNJSuxW5AZynMF3zzOMEXi
To: /content/data/AGL.pdf
100%|██████████| 14.2M/14.2M [00:00<00:00, 55.8MB/s]
Downloading...
From: https://drive.google.com/uc?id=1eZvP7zNSs9SqlMyNeucuurMGp-B4qwSw
To: /content/data/ANZ
100%|██████████| 9.30M/9.30M [00:00<00:00, 18.7MB/s]
Downloading...
From: https://drive.google.com/uc?id=17Og3VhIHgiqByuqCXdiXVoA_jtI15Hxz
To: /content/data/CBA.pdf
100%|██████████| 16.0M/16.0M [00:00<00:00, 65.2MB/s]
Downloading...
From: https://drive.google.com/uc?id=1EVdVxcyJuHUEuK9Y4VLMAJvk-dwwuilc
To: /content/data/Coles Group.pdf
100%|██████████| 8.40M/8.40M [00:00<00:00, 29.5MB/s]
Downloading...
From: https://drive.google.com/uc?id=1BQGJFj2dCAiHCQgJRea9YFn-gxbgQlXF
To: /content/data/combined.csv
100%|██████████| 4.22M/4.22M [00:00<00:00, 29.7MB/s]
Downloading...
From: https://drive.google.com/uc?id=1MnbN2U-o

GPU available: True
Setup complete ✓


In [2]:
# --- PART 1: ESG PDF EXTRACTION ---
# CHANGE: paragraph-aware chunking with 50-word overlap between chunks
# Preserves sentence/paragraph context that hard 400-word splits destroyed
# Also reports pages extracted vs total to catch scanned PDFs early

import pdfplumber

filename_to_company = {
    'Woolworths Australia': 'Woolworths Australia',
    'UniQlo':               'UniQlo',
    'Santos.pdf':           'Santos',
    'Qantas':               'Qantas',
    'NBN.pdf':              'NBN',
    'Muji.pdf':             'Muji',
    'LVMH.pdf':             'LVMH',
    'Gucci.pdf':            'Gucci',
    'Coles Group.pdf':      'Coles Group',
    'CBA.pdf':              'CBA',
    'ANZ':                  'ANZ',
    'AGL.pdf':              'AGL',
}

CHUNK_SIZE = 400   # target words per chunk
OVERLAP    = 50    # word overlap between consecutive chunks
MIN_CHUNK  = 50    # discard anything shorter than this

def paragraph_chunks(full_text, chunk_size=CHUNK_SIZE, overlap=OVERLAP, min_len=MIN_CHUNK):
    """
    Paragraph-aware chunking with overlap.
    - Splits on double newlines first to respect natural paragraph boundaries
    - Accumulates paragraphs into ~400 word chunks
    - Uses 50-word overlap so claims split across boundaries are preserved
    """
    paragraphs = [p.strip() for p in re.split(r'\n{2,}', full_text) if p.strip()]

    chunks = []
    current_words = []

    for para in paragraphs:
        para_words = para.split()

        # if adding this paragraph stays under chunk_size, keep accumulating
        if len(current_words) + len(para_words) <= chunk_size:
            current_words.extend(para_words)
        else:
            # flush current buffer
            if len(current_words) >= min_len:
                chunks.append(' '.join(current_words))

            # if single paragraph is itself longer than chunk_size, split with overlap
            if len(para_words) > chunk_size:
                for i in range(0, len(para_words), chunk_size - overlap):
                    sub = para_words[i:i + chunk_size]
                    if len(sub) >= min_len:
                        chunks.append(' '.join(sub))
                current_words = []
            else:
                # start new buffer — carry overlap from last chunk for context
                if chunks:
                    carry = chunks[-1].split()[-overlap:]
                    current_words = carry + para_words
                else:
                    current_words = para_words

    # flush remainder
    if len(current_words) >= min_len:
        chunks.append(' '.join(current_words))

    return chunks

all_chunks = []

for filename, company_name in filename_to_company.items():
    filepath = os.path.join(DATA_DIR, filename)

    if not os.path.exists(filepath):
        print(f'MISSING: {filename} — skipping')
        continue

    full_text = ''
    pages_extracted = 0

    with pdfplumber.open(filepath) as pdf:
        total_pages = len(pdf.pages)
        for page in pdf.pages:
            text = page.extract_text()
            if text:
                full_text += text + '\n\n'  # to preserve paragraph boundaries
                pages_extracted += 1

    if not full_text.strip():
        print(f'{company_name}: WARNING — 0/{total_pages} pages extracted (likely scanned PDF, needs OCR)')
        continue

    chunks = paragraph_chunks(full_text)
    word_count = len(full_text.split())

    for chunk in chunks:
        all_chunks.append({
            'company':     company_name,
            'source_type': 'esg_report',
            'text':        chunk
        })

    print(f'{company_name}: {pages_extracted}/{total_pages} pages | {word_count} words → {len(chunks)} chunks')

esg_df = pd.DataFrame(all_chunks)
esg_df.to_csv(f'{OUTPUT_DIR}/esg_chunks.csv', index=False)
print(f'\nDone — {len(esg_df)} total ESG chunks saved')

Woolworths Australia: 88/89 pages | 75574 words → 249 chunks
UniQlo: 48/48 pages | 26583 words → 92 chunks
Santos: 286/286 pages | 141190 words → 491 chunks
Qantas: 132/132 pages | 54293 words → 202 chunks
NBN: 36/36 pages | 11146 words → 40 chunks
Muji: 48/48 pages | 58365 words → 186 chunks
LVMH: 151/162 pages | 53393 words → 187 chunks
Gucci: 62/62 pages | 6814 words → 24 chunks
Coles Group: 75/75 pages | 40158 words → 143 chunks
CBA: 228/229 pages | 282720 words → 893 chunks
ANZ: 62/62 pages | 32731 words → 113 chunks
AGL: 232/232 pages | 114962 words → 418 chunks

Done — 3038 total ESG chunks saved


In [3]:
# --- PART 2: NEWS ARTICLE SCRAPING ---
# Fetches sustainability-related news for each company via NewsAPI
#
# FRAMING NOTE: News is treated as EXTERNAL NARRATIVE, not ground truth.
# ESG reports = how the company sees itself (self-reported)
# News        = how the world perceives the company (external)
# The divergence between these two signals is our greenwashing proxy.
#
# TIME MISMATCH is accepted as a feature:
# Reports are 2023/2024 annual documents.
# News is current (2025/2026). This lets us ask:
# "Did the company's commitments translate into sustained external recognition?"
#
# Free tier: 30 articles/company max, last 30 days only, 1 req/sec

from newsapi import NewsApiClient
import time

api = NewsApiClient(api_key='822e469b13934cf2adc43ce8872be994')

# More specific queries to reduce cross-company name contamination
# e.g. "AGL Energy" instead of "AGL" to avoid US stock ticker confusion
company_queries = {
    'AGL':                  'AGL Energy sustainability OR environment OR ESG OR emissions',
    'CBA':                  'Commonwealth Bank sustainability OR environment OR ESG OR emissions',
    'ANZ':                  'ANZ Bank sustainability OR environment OR ESG OR emissions',
    'Qantas':               'Qantas sustainability OR environment OR ESG OR emissions',
    'Santos':               'Santos oil gas sustainability OR environment OR ESG OR emissions Australia',
    'NBN':                  'NBN Co sustainability OR environment OR ESG OR emissions',
    'Muji':                 'Muji retail sustainability OR environment OR ESG OR emissions',
    'LVMH':                 'LVMH luxury sustainability OR environment OR ESG OR emissions',
    'UniQlo':               'Uniqlo fashion sustainability OR environment OR ESG OR emissions',
    'Woolworths Australia': 'Woolworths Australia sustainability OR environment OR ESG OR emissions',
    'Coles Group':          'Coles Group sustainability OR environment OR ESG OR emissions',
    'Gucci':                'Gucci fashion sustainability OR environment OR ESG OR emissions',
}

all_articles = []

for company, query in company_queries.items():
    response = api.get_everything(
        q=query,
        language='en',
        page_size=30,
        sort_by='relevancy'
    )

    for article in response['articles']:
        all_articles.append({
            'company':      company,
            'source_type':  'news',
            'text':         str(article['title']) + '. ' + str(article['description']),
            'published_at': article['publishedAt'],
            'source':       article['source']['name']
        })

    print(f"{company}: {len(response['articles'])} articles fetched")
    time.sleep(1)

news_df = pd.DataFrame(all_articles)
news_df.dropna(subset=['text'], inplace=True)
news_df.to_csv(f'{OUTPUT_DIR}/news_articles.csv', index=False)
print(f'\nDone — {len(news_df)} articles saved')

AGL: 5 articles fetched
CBA: 29 articles fetched
ANZ: 29 articles fetched
Qantas: 28 articles fetched
Santos: 18 articles fetched
NBN: 8 articles fetched
Muji: 2 articles fetched
LVMH: 28 articles fetched
UniQlo: 23 articles fetched
Woolworths Australia: 30 articles fetched
Coles Group: 28 articles fetched
Gucci: 30 articles fetched

Done — 258 articles saved


In [4]:
# --- PART 3: CLEAN + COMBINE ---
# Adds basic text cleaning before combining sources
# Removes URLs, special characters, and very short rows
# Cleaning is lightweight — ClimateBERT handles its own tokenisation internally

def clean_text(text):
    text = str(text)
    text = re.sub(r'http\S+|www\S+', '', text)           # remove URLs
    text = re.sub(r'[^\w\s.,;:!?()\-\'\"]+', ' ', text)  # strip junk chars
    text = re.sub(r'\s+', ' ', text).strip()              # normalise whitespace
    return text if len(text.split()) >= 15 else ''        # drop near-empty rows

esg_df  = pd.read_csv(f'{OUTPUT_DIR}/esg_chunks.csv')
news_df = pd.read_csv(f'{OUTPUT_DIR}/news_articles.csv')

esg_df['text']  = esg_df['text'].apply(clean_text)
news_df['text'] = news_df['text'].apply(clean_text)

esg_df  = esg_df[esg_df['text'].str.len() > 0].reset_index(drop=True)
news_df = news_df[news_df['text'].str.len() > 0].reset_index(drop=True)

news_df = news_df[['company', 'source_type', 'text']]

combined_df = pd.concat([esg_df, news_df], ignore_index=True)
combined_df.to_csv(f'{OUTPUT_DIR}/combined.csv', index=False)

print(f'ESG chunks:     {len(esg_df)}')
print(f'News articles:  {len(news_df)}')
print(f'Total combined: {len(combined_df)}')
print(combined_df['source_type'].value_counts())
print('\nPer company:')
print(combined_df.groupby(['company','source_type']).size().unstack(fill_value=0))

ESG chunks:     3038
News articles:  254
Total combined: 3292
source_type
esg_report    3038
news           254
Name: count, dtype: int64

Per company:
source_type           esg_report  news
company                               
AGL                          418     5
ANZ                          113    28
CBA                          893    29
Coles Group                  143    28
Gucci                         24    29
LVMH                         187    28
Muji                         186     2
NBN                           40     6
Qantas                       202    28
Santos                       491    18
UniQlo                        92    23
Woolworths Australia         249    30


In [5]:
# --- PART 3: CLEAN + COMBINE ---
# Adds basic text cleaning before combining sources
# Removes URLs, special characters, and very short rows
# Cleaning is lightweight — ClimateBERT handles its own tokenisation internally

def clean_text(text):
    text = str(text)
    text = re.sub(r'http\S+|www\S+', '', text)           # remove URLs
    text = re.sub(r'[^\w\s.,;:!?()\-\'\"]+', ' ', text)  # strip junk chars
    text = re.sub(r'\s+', ' ', text).strip()              # normalise whitespace
    return text if len(text.split()) >= 15 else ''        # drop near-empty rows

esg_df  = pd.read_csv(f'{OUTPUT_DIR}/esg_chunks.csv')
news_df = pd.read_csv(f'{OUTPUT_DIR}/news_articles.csv')

esg_df['text']  = esg_df['text'].apply(clean_text)
news_df['text'] = news_df['text'].apply(clean_text)

esg_df  = esg_df[esg_df['text'].str.len() > 0].reset_index(drop=True)
news_df = news_df[news_df['text'].str.len() > 0].reset_index(drop=True)

news_df = news_df[['company', 'source_type', 'text']]

combined_df = pd.concat([esg_df, news_df], ignore_index=True)
combined_df.to_csv(f'{OUTPUT_DIR}/combined.csv', index=False)

print(f'ESG chunks:     {len(esg_df)}')
print(f'News articles:  {len(news_df)}')
print(f'Total combined: {len(combined_df)}')
print(combined_df['source_type'].value_counts())
print('\nPer company:')
print(combined_df.groupby(['company','source_type']).size().unstack(fill_value=0))

ESG chunks:     3038
News articles:  254
Total combined: 3292
source_type
esg_report    3038
news           254
Name: count, dtype: int64

Per company:
source_type           esg_report  news
company                               
AGL                          418     5
ANZ                          113    28
CBA                          893    29
Coles Group                  143    28
Gucci                         24    29
LVMH                         187    28
Muji                         186     2
NBN                           40     6
Qantas                       202    28
Santos                       491    18
UniQlo                        92    23
Woolworths Australia         249    30


In [6]:
# --- PUSH OUTPUTS TO GITHUB ---
import shutil

!git clone https://{GITHUB_TOKEN}@github.com/{REPO}.git {REPO_DIR} -q 2>/dev/null || echo "Repo already cloned"

processed_dir = f'{REPO_DIR}/data/processed'
os.makedirs(processed_dir, exist_ok=True)

for f in os.listdir(OUTPUT_DIR):
    if f.endswith('.csv'):
        shutil.copy(f'{OUTPUT_DIR}/{f}', f'{processed_dir}/{f}')
        print(f'Copied {f}')

!cd {REPO_DIR} && git config user.email "{GIT_EMAIL}"
!cd {REPO_DIR} && git config user.name "{GIT_NAME}"
!cd {REPO_DIR} && git add data/processed/
!cd {REPO_DIR} && git commit -m "update preprocessed data outputs"
!cd {REPO_DIR} && git push

print('\nAll outputs pushed to GitHub → data/processed/')

Copied esg_chunks.csv
Copied news_articles.csv
Copied combined.csv
[main f0c889c] update preprocessed data outputs
 3 files changed, 6448 insertions(+), 3710 deletions(-)
Enumerating objects: 13, done.
Counting objects: 100% (13/13), done.
Delta compression using up to 2 threads
Compressing objects: 100% (6/6), done.
Writing objects: 100% (7/7), 706.07 KiB | 2.05 MiB/s, done.
Total 7 (delta 5), reused 0 (delta 0), pack-reused 0
remote: Resolving deltas: 100% (5/5), completed with 4 local objects.
To https://github.com/Swas26/NLP-ESG.git
   3fabdad..f0c889c  main -> main

All outputs pushed to GitHub → data/processed/
